In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, first install it from [https://ollama.com/download](https://ollama.com/download) for your operating system.\n\n- **macOS**: download and install the `.pkg`\n- **Windows**: download and install the `.msi`\n- **Linux**: run:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nAfter installation, open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\nTo check that the local server is running, you can also use:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you need to restart the Ollama server during homework, use:\n\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about **Olama** specifically.\n\nIf you mean **running the course locally**, the FAQ says you can do that instead of using Codespaces, as long as you’re comfortable setting up:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other tools needed for the module\n\nIt also says that if you run locally, you should **document your setup** and keep your environment **reproducible**.\n\nIf you meant something else by “Olama,” let me know and I can check the FAQ context again.'

## Asking without tools

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—you can usually join a course after it’s started, but it depends on the course’s enrollment policy and schedule.\n\nIf you want, I can help you figure out the best next step. A couple quick questions:\n1. Which course is it?\n2. Is it online or in-person?\n3. Do you know whether registration is still open?\n\nIf you’re asking as a message to an instructor or organizer, you could say:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll late? If so, please let me know the next steps.'

### Defining the tool

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered course can I join it late enrollment"}', call_id='call_OY46XaRkFMWSPLzc0FIXXzUg', name='search', type='function_call', id='fc_09f85ee867efbe9b006a38b9b687d0819aa17fbdb76ae8f0d3', namespace=None, status='completed')]

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can still join the course.\n\nIf your goal is just to learn, you can start anytime.  \nIf you want a certificate, make sure you finish while the course is running and submit your project before submissions close.'

## Token usage

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(852, 50)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
